# Composite DIMSIM geometry analysis

Imports the exact checked canonical implementation from `CODE_ROOT`.


In [ ]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '1f77df1526f79a9b67fd092adad79dbbf24b8d79', 'source_sha256': {'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/bundle.py': '0070c4f280ceb0e5802577fa58faf25e107f6e087790b5ba29974f1c0874d41d', 'evaluation/representations.py': '8d41e607938a592559f77a4e3900ddda3c15975017d0589fbad59a634c43ddbb', 'model.py': '9f6bd2de95cec891c875280e518726ec1c9f6f0ec6080b55fe564a88170415af', 'evaluation/encoders.py': '93444a8a740948ca06632cf68792377995fd0d88f6b152f68640e95d21a04aa4', 'evaluation/analysis.py': 'c06233981812e189e8b696c47fbe5d53d665a3d20bb611f0db15dffc12345577', 'evaluation/runner.py': 'a48b4e0478cf021a89f460641e94c559ca5f50f223e3dc7596a3a47e53d81e35'}, 'embedded_sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'generator_sha256': '496db360a59be5f2a418dcf1e37b209409c49dde9154885a1c2a8d5f314b1867'}


In [ ]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["dimsim==0.2.2", "pypinyin==0.55.0", "numpy", "scipy", "pandas", "matplotlib"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("dimsim", "pypinyin", "numpy", "scipy", "pandas", "matplotlib"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("dimsim") != "0.2.2" or importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pinned DIMSIM preparation dependencies do not match")


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    current = current.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root from current path")

CODE_ROOT = find_project_root()
if any(name == "phonetics" or name.startswith("phonetics.") for name in sys.modules):
    raise RuntimeError("Composite evaluation requires a fresh kernel before importing phonetics")
sys.path.insert(0, str(CODE_ROOT / "src"))
from phonetics.evaluation.runner import run_evaluation
from phonetics.evaluation.analysis import compare_paired_runs

for relative, expected in BUILD_METADATA["source_sha256"].items():
    actual = hashlib.sha256((CODE_ROOT / "src" / "phonetics" / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Local source differs from notebook build: {relative}")


In [ ]:
from pathlib import Path

EXPERIMENT_ID = "1.0.0"
SEED = 42
LAMBDA_MSE = 1.0

PROJECT_ROOT = find_project_root()
BUNDLE_PATH = PROJECT_ROOT / "datasets" / "evaluations" / "1.0.0"
OUTPUT_ROOT = PROJECT_ROOT / "datasets" / "evaluation-runs"
EXPERIMENT_PATH = PROJECT_ROOT / "datasets" / "experiments" / EXPERIMENT_ID
RUNS_ROOT = PROJECT_ROOT / "datasets" / "runs" / EXPERIMENT_ID

RUNS = [
    {"run_id": "lambda-0", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "ce_only" / f"seed-{SEED}" / "best.pt"},
    {"run_id": "lambda-1.0", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "combined" / f"lambda-{LAMBDA_MSE:g}" / f"seed-{SEED}" / "best.pt"},
    {"run_id": "random-seed-42", "adapter": "random", "random_seed": 42, "random_dimension": 1024},
]
DEVICE = "cpu"
BATCH_SIZE = 32
PAIRED_COMPARISONS = [("lambda-0", "lambda-1.0")]


In [ ]:
import torch
print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(f"Selected device {DEVICE!r} requires CUDA, but CUDA is unavailable")


In [ ]:
missing_checkpoints = [str(specification["checkpoint_path"]) for specification in RUNS
                       if specification["adapter"] != "random" and not Path(specification["checkpoint_path"]).is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Training checkpoints are required before evaluation: " + ", ".join(missing_checkpoints))

completed = {}
for specification in RUNS:
    arguments = dict(specification)
    run_id = arguments.pop("run_id")
    adapter = arguments.pop("adapter")
    completed[run_id] = run_evaluation(
        BUNDLE_PATH, output_root=OUTPUT_ROOT, run_id=run_id, adapter=adapter,
        device=DEVICE, batch_size=BATCH_SIZE, code_provenance=BUILD_METADATA["source_sha256"], **arguments,
    )

for left, right in PAIRED_COMPARISONS:
    compare_paired_runs(
        completed[left] / "analysis", completed[right] / "analysis",
        output_dir=OUTPUT_ROOT / f"compare-{left}-vs-{right}",
    )

for run_id, path in completed.items():
    metrics = json.loads((path / "analysis" / "metrics.json").read_text(encoding="utf-8"))
    print(run_id, {level: values["overall"] for level, values in metrics.items()})
